<a href="https://colab.research.google.com/github/NikolaiZolotykh/MachineLearningCourse/blob/master/VADII_06_%D0%9E%D1%86%D0%B5%D0%BD%D0%BA%D0%B0_%D1%81%D1%82%D0%BE%D0%B8%D0%BC%D0%BE%D1%81%D1%82%D0%B8_%D0%BA%D0%B2%D0%B0%D1%80%D1%82%D0%B8%D1%80%D1%8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np              # Одномерные и многомерные массивы (array)
import pandas as pd             # Таблицы и временные ряды (dataframe, series)
import matplotlib.pyplot as plt # Научная графика
import seaborn as sns           # Еще больше красивой графики для визуализации данных
import sklearn                  # Алгоритмы машинного обучения

# **Emergency Dispatch & Traffic Physics Dataset**  
# **Набор данных по экстренному реагированию и физики дорожного движения**


Необходимо заполнить таблицу успеваемости данными о выбранном датасете

### 1. Загружаем данные




In [5]:
url = "https://raw.githubusercontent.com/Svetlana-plv/Machine_Learning/refs/heads/main/synthetic_traffic_benchmark_200k.csv"
data_raw = pd.read_csv(url)

### 2. Краткая сводка

In [8]:
data_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 29 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   Record_ID                            200000 non-null  object 
 1   Timestamp                            200000 non-null  object 
 2   Date                                 200000 non-null  object 
 3   Hour                                 200000 non-null  int64  
 4   Month                                200000 non-null  int64  
 5   Day_of_Week                          200000 non-null  object 
 6   Time_of_Day                          200000 non-null  object 
 7   Is_Holiday                           200000 non-null  int64  
 8   Area_Type                            200000 non-null  object 
 9   Road_Type                            200000 non-null  object 
 10  Intersection_Type                    131481 non-null  object 
 11  Speed_Limit_k

### 3. Заполнение таблицы

In [9]:
# ============================================================
# 1. Базовые размеры
# ============================================================
N, d = data_raw.shape
print(f"N (число объектов) = {N}")
print(f"d (число признаков) = {d}")

# ============================================================
# 2. Целевая переменная и K (число классов)
# ============================================================
# Основная целевая переменная для классификации — Accident_Occurred
target = "Accident_Occurred"

print("\nРаспределение классов Accident_Occurred:")
print(data_raw[target].value_counts())
print("\nДоли:")
print(data_raw[target].value_counts(normalize=True) * 100)

K = data_raw[target].nunique()
print(f"\nK (число классов для {target}) = {K}")

# Дополнительно — K для других целевых переменных
for col in ["Accident_Severity", "Dispatch_Priority", "Collision_Type"]:
    if col in data_raw.columns:
        print(f"K для {col} = {data_raw[col].nunique()}")

# ============================================================
# 3. Пропущенные значения (общий процент по всей таблице)
# ============================================================
# Считаем пустые ячейки (NaN) по всей таблице
missing_total_pct = data_raw.isna().mean().mean() * 100
print(f"\nПропуски, % (по всей таблице) = {missing_total_pct:.2f}")

# Пропуски по столбцам
print("\nПропуски по столбцам (только там, где > 0):")
miss_by_col = data_raw.isna().mean() * 100
print(miss_by_col[miss_by_col > 0].sort_values(ascending=False))

# Если считать "N/A" в Dispatch_Priority как пропуск:
if "Dispatch_Priority" in data_raw.columns:
    na_like = (data_raw["Dispatch_Priority"] == "N/A").mean() * 100
    print(f"\nDispatch_Priority == 'N/A': {na_like:.2f}%")

# ============================================================
# 4. Несбалансированность классов
# ============================================================
counts = data_raw[target].value_counts()
ratio = counts.max() / counts.min()
print(f"\nСоотношение классов (max/min) = {ratio:.1f} : 1")
print(f"В процентах: {counts.max()/N*100:.1f}% / {counts.min()/N*100:.1f}%")

# ============================================================
# 5. Выбросы (примерная оценка)
# ============================================================
# Метод IQR для числовых столбцов
numeric_cols = data_raw.select_dtypes(include=[np.number]).columns.tolist()
outlier_mask = pd.Series(False, index=data_raw.index)

for col in numeric_cols:
    q1 = data_raw[col].quantile(0.25)
    q3 = data_raw[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_mask |= (data_raw[col] < lower) | (data_raw[col] > upper)

outliers_pct = outlier_mask.mean() * 100
print(f"\nВыбросы (IQR-метод, хотя бы в одном числовом признаке), % = {outliers_pct:.2f}")

# Дополнительно — явные аномалии
if "Recorded_Speed_kmh" in data_raw.columns:
    print(f"Recorded_Speed_kmh == 999: {(data_raw['Recorded_Speed_kmh'] == 999).sum()} строк")
if "Road_Closure_Duration_min" in data_raw.columns:
    print(f"Road_Closure_Duration_min > 1000: {(data_raw['Road_Closure_Duration_min'] > 1000).sum()} строк")

# ============================================================
# 6. Типы признаков
# ============================================================
num_cols = data_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = data_raw.select_dtypes(include=["object", "category"]).columns.tolist()

# Исключаем явные идентификаторы и datetime из категориальных
id_like = ["Record_ID", "Timestamp", "Date"]
cat_cols_clean = [c for c in cat_cols if c not in id_like]
other_cols = [c for c in data_raw.columns if c in id_like]

print(f"\nЧисловых признаков = {len(num_cols)}")
print(f"Категориальных признаков = {len(cat_cols_clean)}")
print(f"Признаков других типов = {len(other_cols)}: {other_cols}")

# Текстовые признаки — ищем столбцы, где средняя длина строки большая
text_cols = []
for c in cat_cols_clean:
    avg_len = data_raw[c].astype(str).str.len().mean()
    if avg_len > 30:  # порог: если средняя длина строки > 30 символов — вероятно, текст
        text_cols.append(c)

print(f"Текстовых признаков = {len(text_cols)}: {text_cols}")

# ============================================================
# 7. Итоговая таблица
# ============================================================
summary = pd.DataFrame({
    "Параметр": [
        "N (число объектов)",
        "d (число признаков)",
        "K (число классов)",
        "Пропуски, %",
        "Несбалансированные классы (соотношение)",
        "Выбросы, %",
        "Числовых признаков",
        "Категориальных признаков",
        "Текстовых признаков",
        "Признаков других типов",
    ],
    "Значение": [
        N,
        d,
        K,
        round(missing_total_pct, 2),
        f"{counts.max()/N*100:.1f}/{counts.min()/N*100:.1f}",
        round(outliers_pct, 2),
        len(num_cols),
        len(cat_cols_clean),
        len(text_cols),
        f"{len(other_cols)} ({', '.join(other_cols)})",
    ],
})

print("\n" + "="*60)
print("ИТОГОВАЯ ТАБЛИЦА")
print("="*60)
print(summary.to_string(index=False))

N (число объектов) = 200000
d (число признаков) = 29

Распределение классов Accident_Occurred:
Accident_Occurred
0    191769
1      8231
Name: count, dtype: int64

Доли:
Accident_Occurred
0    95.8845
1     4.1155
Name: proportion, dtype: float64

K (число классов для Accident_Occurred) = 2
K для Accident_Severity = 5
K для Dispatch_Priority = 3
K для Collision_Type = 4

Пропуски, % (по всей таблице) = 7.95

Пропуски по столбцам (только там, где > 0):
Dispatch_Priority        95.8845
Collision_Type           95.8845
Intersection_Type        34.2595
Visibility_m              2.5465
Driver_Experience_yrs     1.9980
dtype: float64

Dispatch_Priority == 'N/A': 0.00%

Соотношение классов (max/min) = 23.3 : 1
В процентах: 95.9% / 4.1%

Выбросы (IQR-метод, хотя бы в одном числовом признаке), % = 18.19
Recorded_Speed_kmh == 999: 104 строк
Road_Closure_Duration_min > 1000: 62 строк

Числовых признаков = 15
Категориальных признаков = 11
Признаков других типов = 3: ['Record_ID', 'Timestamp', 'Dat